# JEPA-for-Trading V2

One-shot JEPA world model with VICReg, action-conditioned portfolio imagination, and an MPC-style planner.

In [ ]:
import os
import sys
from pathlib import Path

REPO = Path('/kaggle/working/jepa-for-trading')
if not REPO.exists():
    !git clone -b version2 https://github.com/aurvl/jepa-for-trading.git {REPO}
%cd /kaggle/working/jepa-for-trading
!{sys.executable} -m pip install -q -e . --no-deps

SRC = str(REPO / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import jepa_trading
print('jepa_trading loaded from:', jepa_trading.__file__)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from jepa_trading.config import ensure_dirs, load_config
from jepa_trading.data.pipeline import prepare_market_data, create_v2_dataloaders
from jepa_trading.data.v2_dataset import PortfolioActionConfig
from jepa_trading.models.world_model_v2 import V2WorldModel
from jepa_trading.training.train_v2 import train_v2_world_model
from jepa_trading.training.checkpoints import load_checkpoint
from jepa_trading.planning.v2_planner import V2ImaginationPlanner
from jepa_trading.evaluation.v2_backtest import run_v2_planner_backtest
from jepa_trading.evaluation.backtest import run_weight_strategy, buy_and_hold_weight, equal_weight, momentum_weight, volatility_target_weight, random_long_only_weight
from jepa_trading.evaluation.metrics import metrics_table
from jepa_trading.evaluation.plots import plot_equity_curves, plot_drawdown, plot_turnover
from jepa_trading.evaluation.statistical_tests import randomization_p_value, bootstrap_mean_return_p_value
from jepa_trading.rl.env import TradingEnv
from jepa_trading.rl.observer import RawMarketObserver
from jepa_trading.utils.device import get_device
from jepa_trading.utils.seed import seed_everything

In [ ]:
config = load_config('configs/default.yaml')

macro_candidates = []
if Path('/kaggle/input').exists():
    macro_candidates.extend(Path('/kaggle/input').rglob('macro_data.parquet'))
    macro_candidates.extend(Path('/kaggle/input').rglob('estimated_volatility_with_macro.csv'))
    macro_candidates.extend(Path('/kaggle/input').rglob('*macro*.csv'))
explicit_macro_csv = Path('/kaggle/input/datasets/aurelvehi/estimated-volatility-with-macro/estimated_volatility_with_macro.csv')
if explicit_macro_csv.exists():
    macro_candidates.insert(0, explicit_macro_csv)
if macro_candidates:
    config['data']['macro_path'] = str(macro_candidates[0])
print('Macro file used:', config['data']['macro_path'])

# Trading modes: long_only, long_short, market_neutral.
config['portfolio']['mode'] = 'long_only'

FAST_DEV_RUN = False
if FAST_DEV_RUN:
    config['data']['tickers'] = config['data']['tickers'][:8]
    config['training']['batch_size'] = 8
    config['v2']['world_max_steps'] = 30
    config['v2']['world_warmup_steps'] = 10
    config['training']['eval_every'] = 10
    config['v2']['planner_action_samples'] = 16

ensure_dirs(config)
seed_everything(config['seed'])
device = get_device(config['device'])
device

In [ ]:
prepared_df, arrays, feature_columns = prepare_market_data(config, force_download=False)
loaders = create_v2_dataloaders(config, arrays)

sample_batch = next(iter(loaders['train']))
portfolio_state_dim = sample_batch['portfolio_state'].shape[-1]
action_dim = sample_batch['action'].shape[-1]

print('rows:', len(prepared_df))
print('dates:', arrays.dates.min(), '->', arrays.dates.max())
print('assets:', len(arrays.tickers), arrays.tickers)
print('features:', len(feature_columns), feature_columns)
print('portfolio_state_dim:', portfolio_state_dim, 'action_dim:', action_dim)
print('dataset sizes:', {k: len(v.dataset) for k, v in loaders.items()})

In [ ]:
model = V2WorldModel(
    n_features=len(feature_columns),
    max_assets=len(arrays.tickers),
    portfolio_state_dim=portfolio_state_dim,
    action_dim=action_dim,
    **config['model'],
    ema_decay=config['training']['ema_decay'],
    hidden_dim=config['v2']['hidden_dim'],
    policy_mode=config['portfolio']['mode'],
    max_abs_weight=config['portfolio']['max_long_weight'],
)

v2_ckpt = Path(config['training']['checkpoint_dir']) / 'v2_world_model.pt'
if v2_ckpt.exists():
    print('Loading V2 world model checkpoint:', v2_ckpt)
    load_checkpoint(v2_ckpt, model, map_location=device)
    v2_history = pd.DataFrame()
else:
    v2_history = train_v2_world_model(
        model=model,
        train_loader=loaders['train'],
        val_loader=loaders['val'],
        device=device,
        max_steps=config['v2']['world_max_steps'],
        warmup_steps=config['v2']['world_warmup_steps'],
        lr=config['training']['lr'],
        weight_decay=config['training']['weight_decay'],
        checkpoint_path=v2_ckpt,
        weights=config['v2']['loss_weights'],
        eval_every=config['training']['eval_every'],
        log_every=config['training']['log_every'],
    )
    load_checkpoint(v2_ckpt, model, map_location=device)
v2_history.tail()

In [ ]:
portfolio_cfg = config['portfolio']
action_cfg = PortfolioActionConfig(
    mode=portfolio_cfg['mode'],
    cash_initial=portfolio_cfg['cash_initial'],
    transaction_cost_bps=portfolio_cfg['transaction_cost_bps'],
    max_long_weight=portfolio_cfg['max_long_weight'],
    max_short_weight=portfolio_cfg['max_short_weight'],
    max_gross_exposure=portfolio_cfg['max_gross_exposure'],
    max_net_exposure=portfolio_cfg['max_net_exposure'],
    borrow_cost_bps=portfolio_cfg['borrow_cost_bps'],
    n_action_samples=config['v2']['planner_action_samples'],
)
planner = V2ImaginationPlanner(
    model=model,
    action_config=action_cfg,
    horizons=config['data']['horizons'],
    n_action_samples=config['v2']['planner_action_samples'],
    device=device,
    seed=config['seed'],
)

test_start = pd.Timestamp(config['data']['val_end']) + pd.Timedelta(days=1)
test_end = arrays.dates.max()
agent_hist = run_v2_planner_backtest(
    arrays=arrays,
    planner=planner,
    start_date=test_start,
    end_date=test_end,
    lookback=config['data']['lookback'],
    cash_initial=config['portfolio']['cash_initial'],
    transaction_cost_bps=config['portfolio']['transaction_cost_bps'],
    max_weight_per_asset=config['portfolio']['max_weight_per_asset'],
    max_turnover=config['portfolio']['max_turnover'],
    mode=config['portfolio']['mode'],
    max_long_weight=config['portfolio']['max_long_weight'],
    max_short_weight=config['portfolio']['max_short_weight'],
    max_gross_exposure=config['portfolio']['max_gross_exposure'],
    max_net_exposure=config['portfolio']['max_net_exposure'],
    borrow_cost_bps=config['portfolio']['borrow_cost_bps'],
)
agent_hist.tail()


In [ ]:
observer = RawMarketObserver(arrays, config['data']['lookback'])
def make_test_env():
    return TradingEnv(
        arrays,
        observer,
        start_date=test_start,
        end_date=test_end,
        lookback=config['data']['lookback'],
        cash_initial=config['portfolio']['cash_initial'],
        transaction_cost_bps=config['portfolio']['transaction_cost_bps'],
        max_weight_per_asset=config['portfolio']['max_weight_per_asset'],
        max_turnover=config['portfolio']['max_turnover'],
    )

bh_hist = run_weight_strategy(make_test_env(), buy_and_hold_weight)
equal_hist = run_weight_strategy(make_test_env(), equal_weight)
mom_hist = run_weight_strategy(make_test_env(), momentum_weight)
vol_hist = run_weight_strategy(make_test_env(), volatility_target_weight)
rng = np.random.default_rng(config['seed'])
random_hists = [
    run_weight_strategy(make_test_env(), lambda env, mask, rng=rng: random_long_only_weight(env, mask, rng))
    for _ in range(100)
]

histories = {
    'V2 JEPA Planner': agent_hist,
    'Buy & Hold': bh_hist,
    'Equal Weight': equal_hist,
    'Momentum': mom_hist,
    'Vol Target': vol_hist,
}
metrics_table(histories)

In [ ]:
plot_equity_curves(agent_hist, bh_hist, random_hists, extra={'Equal Weight': equal_hist, 'Momentum': mom_hist, 'Vol Target': vol_hist})
plot_drawdown(agent_hist, label='V2 JEPA Planner')
plot_turnover(agent_hist)
print('Randomization test:', randomization_p_value(agent_hist, random_hists))
print('Bootstrap vs Buy & Hold:', bootstrap_mean_return_p_value(agent_hist, bh_hist))
agent_hist[['date', 'equity', 'planned_horizon', 'planner_score', 'predicted_log_return', 'predicted_drawdown']].tail()